# Assignment 4 — EvaLoop code ↔ pseudocode

Run on a **fresh Colab GPU runtime** (Runtime → Change runtime type → T4 GPU).
Upload `colab_bundle.zip` below. No API keys are needed.

**Do not mount Google Drive, connect secrets, or upload personal files.** Generated code is tested by EvalPlus in child processes with guards/timeouts inside this disposable VM. These are not a hardened sandbox. Keep the VM disposable and download checkpoints regularly.

Four experiments run on the same 100 tasks, with at most 10 cycles each. Allow hours depending on hardware and survival. Download results after every run. The final ZIP is for finishing the report, not a completed Gradescope submission.

In [ ]:
from google.colab import files
from pathlib import Path
import os, zipfile, subprocess, sys
assert not Path('/content/drive').exists(), 'Use a fresh runtime without Drive'
uploaded = files.upload()  # Choose colab_bundle.zip
assert 'colab_bundle.zip' in uploaded
with zipfile.ZipFile('/content/colab_bundle.zip') as z:
    for name in z.namelist():
        target = (Path('/content') / name).resolve()
        assert target.is_relative_to(Path('/content')), 'Unsafe ZIP member'
    z.extractall('/content')
os.chdir('/content/evaloop_assignment4')
print(Path.cwd())

In [ ]:
# Replacement for the failed setup cell. Run after the ZIP-upload cell.
import os
import subprocess
import sys
from pathlib import Path

os.chdir('/content/evaloop_assignment4')

def checked(command):
    print('+', ' '.join(command), flush=True)
    with subprocess.Popen(command, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError('Setup failed; copy the installation output printed above.')

# uv supplies Python itself, without relying on Colab's venv/ensurepip.
UV_DIR = '/content/evaloop_uv'
checked([sys.executable, '-m', 'pip', 'install', '--target', UV_DIR, 'uv==0.6.17'])
UV = UV_DIR + '/bin/uv'
ENV = '/content/evaloop_py312'
PYTHON = ENV + '/bin/python'
if not Path(PYTHON).exists():
    checked([UV, 'venv', '--python', '3.12.9', '--seed', ENV])
checked([PYTHON, '-m', 'pip', 'install', 'torch==2.6.0',
         '--index-url', 'https://download.pytorch.org/whl/cu124'])
checked([PYTHON, '-m', 'pip', 'install', '-r', 'assignment4/requirements-colab.txt'])
checked([PYTHON, '-m', 'pytest', 'tests/test_assignment4.py', '-q'])
checked([PYTHON, '-c',
         'import sys, torch; '
         'from evaloop.evaluation.py_testing import EvalPlusCodeTester; '
         'from transformers import Qwen2ForCausalLM; '
         'print(sys.version); print(torch.__version__); '
         'assert torch.cuda.is_available(), "Select a GPU runtime in Colab"; '
         'print(torch.cuda.get_device_name(0))'])
print('Setup complete. Continue with the next notebook cell.')


In [ ]:
# Optional recovery after a previous runtime was lost.
RESTORE = False
if RESTORE:
    restored = files.upload()  # Choose the most recent experiment_results.zip
    archive = next(iter(restored))
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            if member.filename.startswith('results/') and not member.is_dir():
                dest = (Path.cwd() / member.filename).resolve()
                assert dest.is_relative_to(Path.cwd() / 'results')
                if dest.exists():
                    raise RuntimeError('Refusing to overwrite a current checkpoint; restore in a fresh runtime')
                dest.parent.mkdir(parents=True, exist_ok=True)
                dest.write_bytes(z.read(member))
    print('Restored checkpoints')

In [ ]:
def download_results():
    target = Path('/content/experiment_results.zip')
    with zipfile.ZipFile(target, 'w', zipfile.ZIP_DEFLATED) as z:
        for folder in ['results', 'analysis']:
            if Path(folder).exists():
                for p in Path(folder).rglob('*'):
                    if p.is_file():
                        z.write(p, str(p))
        frozen = subprocess.check_output([PYTHON, '-m', 'pip', 'freeze'], text=True)
        z.writestr('environment.txt', frozen)
    files.download(str(target))

def run(model, configuration):
    try:
        subprocess.run([PYTHON, '-m', 'assignment4.run', '--model', model,
                        '--configuration', configuration, '--disposable-colab'], check=True)
    finally:
        download_results()  # Also save progress if the process fails.


## Qwen 0.5B — standard
Save the offered ZIP download after this cell finishes.

In [ ]:
run('Qwen/Qwen2.5-Coder-0.5B-Instruct', 'standard')

## Qwen 0.5B — pseudocode
Save the offered ZIP download after this cell finishes.

In [ ]:
run('Qwen/Qwen2.5-Coder-0.5B-Instruct', 'pseudocode')

## Qwen 1.5B — standard
Save the offered ZIP download after this cell finishes.

In [ ]:
run('Qwen/Qwen2.5-Coder-1.5B-Instruct', 'standard')

## Qwen 1.5B — pseudocode
Save the offered ZIP download after this cell finishes.

In [ ]:
run('Qwen/Qwen2.5-Coder-1.5B-Instruct', 'pseudocode')

## Validate and summarize all four runs

In [ ]:
subprocess.run([PYTHON, '-m', 'assignment4.analyze'], check=True)
print(Path('analysis/results.md').read_text())
download_results()

## Next step
Return `experiment_results.zip` to the assignment workspace/chat. The report still needs actual findings and at least 20 manually inspected failures. The archive includes the failure-review worksheet, evidence, all four result JSON files and `summary.csv`. A public fork link is also required before submission.

If a cell fails, send its traceback and the latest downloaded ZIP. Do not invent missing results or treat partial summaries as final.